# 🧠 NLP Project — Insurer Reviews Analysis
**Colonne utilisée** : `avis_en` | **Traduction** : skippée | **GPU** : RTX 6000 Ada 48GB

---
## ♻️ CELLULE RESTORE — À exécuter EN PREMIER si tu relances le kernel
> Si tu as déjà tout entraîné une fois et sauvegardé les checkpoints, mets `RESTORE = True` et exécute cette cellule. Tu sautes directement à la cellule 19 sans rien réentraîner. Si c'est la première fois, mets `RESTORE = False`.

In [ ]:
RESTORE = False  # ← True si checkpoints déjà créés

import os, pickle
import pandas as pd
import numpy as np

if RESTORE and os.path.exists('./checkpoints/train_df.csv'):
    print('🔄 Restauration depuis checkpoints...')
    train_df = pd.read_csv('./checkpoints/train_df.csv')
    val_df   = pd.read_csv('./checkpoints/val_df.csv')
    test_df  = pd.read_csv('./checkpoints/test_df.csv')
    y_train  = np.load('./checkpoints/y_train.npy')
    y_val    = np.load('./checkpoints/y_val.npy')
    y_test   = np.load('./checkpoints/y_test.npy')
    with open('./checkpoints/tfidf_vectorizer.pkl', 'rb') as f:
        tfidf = pickle.load(f)
    with open('./checkpoints/tfidf_results.pkl', 'rb') as f:
        tfidf_results = pickle.load(f)
    with open('./checkpoints/keras_tokenizer.pkl', 'rb') as f:
        keras_tok = pickle.load(f)
    with open('./checkpoints/lda_model.pkl', 'rb') as f:
        lda = pickle.load(f)
    with open('./checkpoints/tfidf_lda_vectorizer.pkl', 'rb') as f:
        tfidf_lda = pickle.load(f)
    with open('./checkpoints/topic_labels.pkl', 'rb') as f:
        topic_labels = pickle.load(f)
    preds_emb     = np.load('./checkpoints/preds_emb.npy')
    preds_lstm    = np.load('./checkpoints/preds_lstm.npy')
    preds_deberta = np.load('./checkpoints/preds_deberta.npy')
    preds_mistral = np.load('./checkpoints/preds_mistral.npy')
    import tensorflow as tf
    emb_model  = tf.keras.models.load_model('./checkpoints/emb_model.keras')
    lstm_model = tf.keras.models.load_model('./checkpoints/lstm_model.keras')
    import torch
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    import os as _os
    _os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    deberta_tokenizer = AutoTokenizer.from_pretrained('./best_deberta_model', use_fast=False)
    deberta_model     = AutoModelForSequenceClassification.from_pretrained('./best_deberta_model')
    deberta_model.to(device)
    from transformers import AutoTokenizer as _AT
    mistral_tokenizer = _AT.from_pretrained('./best_mistral_lora', use_fast=True)
    df = pd.read_csv('./checkpoints/df_full.csv')
    print('✅ Restauration complète — saute directement à la cellule 19')
    print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
else:
    print('🚀 Mode from scratch — continue avec la cellule 1')


---
## ⚙️ CELLULE 1 — Imports globaux
> On importe toutes les librairies dont on a besoin. C'est la boîte à outils complète : pandas/numpy pour les données, torch pour le deep learning, transformers pour les modèles HuggingFace, gensim pour Word2Vec, nltk pour le preprocessing NLP classique, tensorflow/keras pour les réseaux récurrents.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings, os, re, json, pickle
from collections import Counter
from tqdm.auto import tqdm

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.decomposition import LatentDirichletAllocation

import gensim
from gensim.models import Word2Vec
import gensim.downloader as api

import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
    pipeline, BitsAndBytesConfig
)
from datasets import Dataset
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from sentence_transformers import SentenceTransformer

import umap
from scipy.spatial.distance import cosine, euclidean

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout,
    Conv1D, GlobalMaxPooling1D, Bidirectional
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import TensorBoard as TFTensorBoard

nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('punkt_tab', quiet=True)

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device : {device}')
if torch.cuda.is_available():
    print(f'✅ GPU    : {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


---
## 📂 CELLULE 2 — Chargement des données
> On charge le CSV et on explore : combien de lignes, colonnes, valeurs manquantes, distribution des notes, liste des assureurs. C'est l'étape d'exploration initiale — indispensable avant de toucher à quoi que ce soit.

In [ ]:
df = pd.read_csv('reviews_concat_prepared.csv')

print(f'Shape      : {df.shape}')
print(f'Colonnes   : {df.columns.tolist()}')
print(f'\nTypes:\n{df.dtypes}')
print(f'\nValeurs manquantes:\n{df.isnull().sum()}')
print(f'\nDistribution notes:\n{df["note"].value_counts().sort_index()}')
print(f'\nAssureurs  : {df["assureur"].unique()}')
df.head(5)


---
## 🧹 CELLULE 3 — Data Cleaning (2 pts)
> Le data cleaning c'est l'étape où on nettoie le texte brut pour le rendre exploitable. On retire les stopwords (mots très fréquents sans sens comme 'the', 'is', 'a'), on passe tout en minuscules, on enlève la ponctuation et les URLs. Sans ce nettoyage les modèles apprennent du bruit plutôt que du signal.

In [ ]:
df = df.dropna(subset=['avis_en']).reset_index(drop=True)

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['avis_clean'] = df['avis_en'].apply(clean_text)
df['tokens']     = df['avis_clean'].apply(word_tokenize)

print(f'Avant  : {df["avis_en"].iloc[0]}')
print(f'Après  : {df["avis_clean"].iloc[0]}')
print(f'\nLongueur moyenne (mots) : {df["tokens"].apply(len).mean():.1f}')
print(f'Longueur médiane        : {df["tokens"].apply(len).median():.0f}')


---
## 📊 CELLULE 4 — Mots fréquents & N-grammes (2 pts)
> On analyse les mots et groupes de mots les plus fréquents. Un n-gramme c'est une séquence de N mots consécutifs — les bigrammes (2 mots) capturent des expressions comme 'customer service' qu'on raterait mot par mot. La wordcloud visualise ça de façon intuitive.

In [ ]:
all_words   = [w for tokens in df['tokens'] for w in tokens]
word_freq   = Counter(all_words)
top_words   = word_freq.most_common(30)

all_bigrams  = [' '.join(bg) for tokens in df['tokens'] for bg in ngrams(tokens, 2)]
all_trigrams = [' '.join(tg) for tokens in df['tokens'] for tg in ngrams(tokens, 3)]
top_bigrams  = Counter(all_bigrams).most_common(20)
top_trigrams = Counter(all_trigrams).most_common(15)

fig, axes = plt.subplots(2, 2, figsize=(20, 14))

words, counts = zip(*top_words)
axes[0,0].barh(words, counts, color='steelblue')
axes[0,0].set_title('Top 30 mots')
axes[0,0].invert_yaxis()

bg_w, bg_c = zip(*top_bigrams)
axes[0,1].barh(bg_w, bg_c, color='salmon')
axes[0,1].set_title('Top 20 bigrammes')
axes[0,1].invert_yaxis()

tg_w, tg_c = zip(*top_trigrams)
axes[1,0].barh(tg_w, tg_c, color='mediumseagreen')
axes[1,0].set_title('Top 15 trigrammes')
axes[1,0].invert_yaxis()

wc = WordCloud(width=800, height=400, background_color='white', colormap='viridis', max_words=100)
wc.generate(' '.join(all_words))
axes[1,1].imshow(wc, interpolation='bilinear')
axes[1,1].axis('off')
axes[1,1].set_title('WordCloud')

plt.tight_layout()
plt.savefig('word_frequencies.png', dpi=150)
plt.show()
print(f'Top 5 bigrammes  : {top_bigrams[:5]}')
print(f'Top 5 trigrammes : {top_trigrams[:5]}')


---
## 🏷️ CELLULE 5 — Préparation des labels sentiment
> On transforme la note (1 à 5) en 3 classes : négatif (1-2), neutre (3), positif (4-5). C'est une discrétisation — on passe d'une variable continue à une variable catégorielle. 3 classes est plus robuste que 5 car les notes intermédiaires sont souvent ambiguës pour les modèles.

In [ ]:
def note_to_sentiment(note):
    if note <= 2.0:  return 0  # négatif
    elif note == 3.0: return 1  # neutre
    else:             return 2  # positif

df['label']         = df['note'].apply(note_to_sentiment)
df['sentiment_str'] = df['label'].map({0: 'négatif', 1: 'neutre', 2: 'positif'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['label'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0],
    color=['tomato','gold','mediumseagreen'], edgecolor='black'
)
axes[0].set_xticklabels(['Négatif','Neutre','Positif'], rotation=0)
axes[0].set_title('Distribution des sentiments')
df['note'].hist(bins=10, ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Distribution des notes brutes')
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150)
plt.show()
print(df['label'].value_counts().sort_index())


---
## ✂️ CELLULE 6 — Train / Val / Test split
> On découpe le dataset en 3 parties : train (70%) pour entraîner, validation (15%) pour surveiller l'overfitting, test (15%) pour l'évaluation finale honnête. Le paramètre `stratify` garantit les mêmes proportions de classes dans chaque split.

In [ ]:
train_val_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label']
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.176, random_state=42, stratify=train_val_df['label']
)

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

print(f'Train      : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)')
print(f'Validation : {len(val_df)}   ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test       : {len(test_df)}  ({len(test_df)/len(df)*100:.1f}%)')
for name, split in [('Train',train_df),('Val',val_df),('Test',test_df)]:
    d = split['label'].value_counts(normalize=True).sort_index()
    print(f'{name}: neg={d[0]:.2f} neu={d[1]:.2f} pos={d[2]:.2f}')


---
## 🗂️ CELLULE 7 — Topic Modeling LDA (2 pts)
> LDA (Latent Dirichlet Allocation) est un algorithme non supervisé qui découvre automatiquement les thèmes latents dans le corpus. Il suppose que chaque document est un mélange de topics, et chaque topic une distribution de mots. Il va trouver tout seul des thèmes comme 'prix/contrat', 'sinistre', 'service client', etc.

In [ ]:
N_TOPICS = 8

tfidf_lda = TfidfVectorizer(max_features=5000, min_df=3, max_df=0.85)
dtm       = tfidf_lda.fit_transform(df['avis_clean'])

lda = LatentDirichletAllocation(
    n_components=N_TOPICS, max_iter=30,
    learning_method='online', random_state=42, n_jobs=-1
)
lda.fit(dtm)

vocab = tfidf_lda.get_feature_names_out()
topic_labels = [
    'Prix & Contrat', 'Service Client', 'Sinistre & Remboursement',
    'Application Mobile', 'Qualité Générale', 'Réactivité',
    'Résiliation', 'Recommandation'
]

topic_words = {}
for i, topic in enumerate(lda.components_):
    top_idx   = topic.argsort()[-15:][::-1]
    top_terms = [vocab[j] for j in top_idx]
    topic_words[i] = top_terms
    print(f'Topic {i} — {topic_labels[i]}: {top_terms[:8]}')

df['dominant_topic'] = lda.transform(dtm).argmax(axis=1)
df['topic_label']    = df['dominant_topic'].apply(lambda x: topic_labels[x])

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for i, ax in enumerate(axes.flatten()):
    if i < N_TOPICS:
        wc_t = WordCloud(width=300, height=200, background_color='white')
        freq_dict = {vocab[j]: lda.components_[i][j] for j in range(len(vocab))}
        wc_t.generate_from_frequencies(freq_dict)
        ax.imshow(wc_t, interpolation='bilinear')
        ax.set_title(topic_labels[i], fontsize=10)
        ax.axis('off')
plt.suptitle('Topics LDA', fontsize=14)
plt.tight_layout()
plt.savefig('lda_topics.png', dpi=150)
plt.show()
print(df['topic_label'].value_counts())


---
## 🔤 CELLULE 8 — Word2Vec Training (2 pts)
> Word2Vec (Google, 2013) repose sur l'hypothèse distributionnelle : un mot est défini par ses voisins. Le modèle entraîne un réseau de neurones à prédire le contexte d'un mot. En sortie, chaque mot est représenté par un vecteur dense de 300 dimensions — des mots sémantiquement proches ont des vecteurs proches. C'est ce qu'on appelle un embedding.

In [ ]:
sentences = df['tokens'].tolist()

w2v_model = Word2Vec(
    sentences=sentences, vector_size=300, window=5,
    min_count=3, workers=8, sg=1, epochs=20, seed=42
)
w2v_model.save('word2vec_reviews.model')
print(f'Vocabulaire W2V : {len(w2v_model.wv)} mots')

for word in ['insurance', 'price', 'service', 'claim', 'cancel']:
    if word in w2v_model.wv:
        print(f'Proches de "{word}" : {w2v_model.wv.most_similar(word, topn=5)}')


---
## 🌐 CELLULE 9 — GloVe Pre-trained Embeddings (1 pt)
> GloVe (Stanford) est une alternative à Word2Vec. Là où Word2Vec apprend depuis des fenêtres de contexte locales, GloVe exploite les statistiques de co-occurrence globales sur tout le corpus. On charge les vecteurs pré-entraînés sur Wikipedia + Gigaword (6 milliards de tokens) — beaucoup plus riche que notre corpus.

In [ ]:
print('Chargement GloVe glove-wiki-gigaword-300...')
glove_model = api.load('glove-wiki-gigaword-300')
print(f'GloVe chargé — {len(glove_model)} vecteurs dim 300')

for word in ['insurance', 'accident', 'refund', 'customer']:
    if word in glove_model:
        print(f'GloVe proches de "{word}" : {glove_model.most_similar(word, topn=5)}')

if all(w in glove_model for w in ['king','man','woman']):
    res = glove_model.most_similar(positive=['king','woman'], negative=['man'], topn=3)
    print(f'\nAnalogy king - man + woman = {res}')


---
## 📐 CELLULE 10 — Distance Euclidienne & Cosinus (1 pt)
> Deux métriques pour mesurer la similarité entre vecteurs. La distance euclidienne mesure la distance géométrique directe. La similarité cosinus mesure l'angle entre deux vecteurs — elle est plus robuste en NLP car insensible à la norme. Cosinus = 1 : vecteurs identiques | 0 : orthogonaux | -1 : opposés.

In [ ]:
def cosine_similarity(v1, v2): return 1 - cosine(v1, v2)
def euclidean_distance(v1, v2): return euclidean(v1, v2)

word_pairs = [
    ('insurance','policy'), ('insurance','accident'),
    ('good','excellent'),   ('good','terrible'),
    ('price','cost'),       ('price','happiness'),
]
print(f'{"Paire":<35} {"Cosinus":>10} {"Euclidien":>12}')
print('-'*60)
for w1, w2 in word_pairs:
    if w1 in glove_model and w2 in glove_model:
        v1, v2 = glove_model[w1], glove_model[w2]
        print(f'{w1} vs {w2:<25} {cosine_similarity(v1,v2):>10.4f} {euclidean_distance(v1,v2):>12.4f}')


---
## 🔍 CELLULE 11 — Semantic Search SBERT + FAISS (Bonus 2 pts)
> La recherche sémantique dépasse la recherche par mot-clé. On encode chaque avis en vecteur dense avec Sentence-BERT puis on stocke dans FAISS (Facebook AI Similarity Search). Quand on cherche 'bad customer service' on trouve les avis proches même sans ces mots exacts.

In [ ]:
import faiss

print('Chargement Sentence-BERT...')
sbert_model = SentenceTransformer('all-mpnet-base-v2', device=str(device))

print('Encoding des avis...')
review_texts = df['avis_en'].tolist()
embeddings   = sbert_model.encode(
    review_texts, batch_size=256,
    show_progress_bar=True, normalize_embeddings=True
)
np.save('review_embeddings.npy', embeddings)

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype('float32'))
faiss.write_index(index, 'faiss_reviews.index')
print(f'Index FAISS : {index.ntotal} vecteurs dim {dim}')

def semantic_search(query, k=5):
    q_vec = sbert_model.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_vec, k)
    for score, idx in zip(scores[0], indices[0]):
        print(f'  [{score:.3f}] {df["note"].iloc[idx]}⭐ {df["assureur"].iloc[idx]} — {df["avis_en"].iloc[idx][:80]}...')

print('\n=== TEST SEMANTIC SEARCH ===')
semantic_search('terrible customer service very slow', k=3)


---
## 📉 CELLULE 12 — Visualisation Embeddings UMAP + Matplotlib (2 pts)
> Les embeddings sont en 300+ dimensions — impossible à visualiser. UMAP (Uniform Manifold Approximation and Projection) réduit ça en 2D en préservant la structure locale et globale. On voit des clusters de mots sémantiquement proches se former naturellement.

In [ ]:
vocab_viz   = list(w2v_model.wv.key_to_index.keys())[:300]
vectors_viz = np.array([w2v_model.wv[w] for w in vocab_viz])

reducer      = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding_2d = reducer.fit_transform(vectors_viz)

plt.figure(figsize=(16, 12))
plt.scatter(embedding_2d[:,0], embedding_2d[:,1], alpha=0.5, s=15, c='steelblue')

highlight = ['insurance','price','service','claim','cancel','good','bad',
             'excellent','terrible','fast','slow','refund','accident','policy']
for word in highlight:
    if word in vocab_viz:
        idx = vocab_viz.index(word)
        plt.annotate(word, xy=(embedding_2d[idx,0], embedding_2d[idx,1]),
                     fontsize=11, color='crimson', fontweight='bold')
        plt.scatter(embedding_2d[idx,0], embedding_2d[idx,1], s=80, c='crimson', zorder=5)

plt.title('Word2Vec embeddings — UMAP 2D', fontsize=16)
plt.savefig('embeddings_umap.png', dpi=150)
plt.show()


---
## 📋 CELLULE 13 — TensorBoard Embeddings
> TensorBoard est l'outil de visualisation de TensorFlow. Son Embedding Projector permet de naviguer en 3D interactif dans l'espace des embeddings et de voir les clusters. On exporte les vecteurs Word2Vec dans le format TensorBoard.

In [ ]:
os.makedirs('tensorboard_logs/embeddings', exist_ok=True)
tb_writer = SummaryWriter('tensorboard_logs/embeddings')

viz_words   = list(w2v_model.wv.key_to_index.keys())[:500]
viz_vectors = torch.tensor(
    np.array([w2v_model.wv[w] for w in viz_words]), dtype=torch.float32
)
tb_writer.add_embedding(viz_vectors, metadata=viz_words, tag='Word2Vec_embeddings')
tb_writer.close()

print('✅ TensorBoard embeddings sauvegardés')
print('   tensorboard --logdir=tensorboard_logs/embeddings --port=6006')


---
## 🤖 CELLULE 14 — Modèle 1 : TF-IDF + ML classique (2 pts)
> TF-IDF (Term Frequency - Inverse Document Frequency) vectorise le texte : TF mesure la fréquence d'un mot dans un document, IDF pénalise les mots présents partout. Le résultat est un vecteur creux (sparse). On compare Logistic Regression, SVM et Random Forest — ces classifieurs classiques servent de baseline.

In [ ]:
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
X_train_tfidf = tfidf.fit_transform(train_df['avis_clean'])
X_val_tfidf   = tfidf.transform(val_df['avis_clean'])
X_test_tfidf  = tfidf.transform(test_df['avis_clean'])

tfidf_models = {
    'Logistic Regression': LogisticRegression(max_iter=500, C=1.0, random_state=42),
    'Linear SVM':          LinearSVC(max_iter=2000, C=1.0, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
}
tfidf_results = {}
for name, clf in tfidf_models.items():
    clf.fit(X_train_tfidf, y_train)
    preds = clf.predict(X_test_tfidf)
    acc   = accuracy_score(y_test, preds)
    f1    = f1_score(y_test, preds, average='weighted')
    tfidf_results[name] = {'accuracy': acc, 'f1': f1, 'preds': preds}
    print(f'{name}: Acc={acc:.4f} | F1={f1:.4f}')

best_tfidf = max(tfidf_results, key=lambda k: tfidf_results[k]['f1'])
print(f'\nMeilleur TF-IDF : {best_tfidf}')
print(classification_report(y_test, tfidf_results[best_tfidf]['preds'],
                             target_names=['négatif','neutre','positif']))


---
## 🧱 CELLULE 15 — Modèle 2 : CNN + Embedding Layer from scratch (2 pts + 1 pt TensorBoard)
> On construit un réseau CNN avec une couche d'Embedding apprise from scratch. L'embedding layer produit des vecteurs denses initialisés aléatoirement et appris pendant l'entraînement. Le CNN extrait des features locales via des filtres glissants. C'est la brique fondamentale de tout modèle NLP moderne.

In [ ]:
MAX_VOCAB  = 20000
MAX_LEN    = 128
EMBED_DIM  = 128
BATCH_SIZE = 256

keras_tok = KerasTokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
keras_tok.fit_on_texts(train_df['avis_clean'])

def encode_keras(texts):
    seqs = keras_tok.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_tr = encode_keras(train_df['avis_clean'])
X_vl = encode_keras(val_df['avis_clean'])
X_te = encode_keras(test_df['avis_clean'])

emb_model = Sequential([
    Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN, name='embedding_layer'),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])
emb_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
emb_model.summary()

os.makedirs('tensorboard_logs/embedding_model', exist_ok=True)
tb_emb = TFTensorBoard(log_dir='tensorboard_logs/embedding_model', histogram_freq=1)

history_emb = emb_model.fit(
    X_tr, y_train, validation_data=(X_vl, y_val),
    epochs=10, batch_size=BATCH_SIZE, callbacks=[tb_emb], verbose=1
)

preds_emb = emb_model.predict(X_te).argmax(axis=1)
print(f'CNN Emb — Acc: {accuracy_score(y_test, preds_emb):.4f} | F1: {f1_score(y_test, preds_emb, average="weighted"):.4f}')
print('TensorBoard : tensorboard --logdir=tensorboard_logs/embedding_model')


---
## 🔁 CELLULE 16 — Modèle 3 : Bi-LSTM + GloVe pré-entraîné (2 pts + 1 pt TensorBoard)
> Bi-LSTM = Bidirectional Long Short-Term Memory. Un LSTM mémorise les dépendances à long terme dans une séquence. Le 'Bi' signifie qu'on le fait tourner dans les deux sens (gauche→droite ET droite→gauche) pour capturer le contexte complet. On initialise les embeddings avec GloVe pré-entraîné au lieu de partir de zéro.

In [ ]:
glove_dim    = 300
embed_matrix = np.zeros((MAX_VOCAB, glove_dim))
n_found      = 0
for word, idx in keras_tok.word_index.items():
    if idx < MAX_VOCAB and word in glove_model:
        embed_matrix[idx] = glove_model[word]
        n_found += 1
print(f'Mots GloVe trouvés : {n_found}/{min(MAX_VOCAB, len(keras_tok.word_index))} '
      f'({n_found/min(MAX_VOCAB, len(keras_tok.word_index))*100:.1f}%)')

lstm_model = Sequential([
    Embedding(MAX_VOCAB, glove_dim, weights=[embed_matrix],
              input_length=MAX_LEN, trainable=True, name='glove_embedding'),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3)),
    Bidirectional(LSTM(64, dropout=0.3)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

os.makedirs('tensorboard_logs/lstm_model', exist_ok=True)
tb_lstm = TFTensorBoard(log_dir='tensorboard_logs/lstm_model', histogram_freq=1)

history_lstm = lstm_model.fit(
    X_tr, y_train, validation_data=(X_vl, y_val),
    epochs=12, batch_size=128, callbacks=[tb_lstm], verbose=1
)

preds_lstm = lstm_model.predict(X_te).argmax(axis=1)
print(f'Bi-LSTM+GloVe — Acc: {accuracy_score(y_test, preds_lstm):.4f} | F1: {f1_score(y_test, preds_lstm, average="weighted"):.4f}')
print('TensorBoard : tensorboard --logdir=tensorboard_logs/lstm_model')


---
## ⚡ CELLULE 17a — DeBERTa-v3-large : Tokenizer + Datasets
> DeBERTa améliore BERT avec une attention disentangled qui sépare contenu et position des tokens, et un enhanced mask decoder. La version large a 400M paramètres. On fait du fine-tuning : on part des poids pré-entraînés sur des centaines de GB de texte et on adapte à notre tâche.

In [ ]:
import sentencepiece
print(f'✅ sentencepiece {sentencepiece.__version__}')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

DEBERTA_NAME = 'microsoft/deberta-v3-large'

deberta_tokenizer = AutoTokenizer.from_pretrained(
    DEBERTA_NAME, use_fast=False, local_files_only=True
)
print(f'✅ Tokenizer : {type(deberta_tokenizer).__name__}')

def hf_tokenize(batch):
    return deberta_tokenizer(
        batch['text'], truncation=True, max_length=256,
        padding=False, return_token_type_ids=False
    )

def df_to_hf_dataset(dataframe):
    ds = Dataset.from_dict({
        'text':  dataframe['avis_en'].astype(str).tolist(),
        'label': dataframe['label'].astype(int).tolist()
    })
    return ds.map(hf_tokenize, batched=True, batch_size=128,
                  remove_columns=['text'], desc='Tokenizing')

train_hf = df_to_hf_dataset(train_df)
val_hf   = df_to_hf_dataset(val_df)
test_hf  = df_to_hf_dataset(test_df)

train_hf.set_format('torch')
val_hf.set_format('torch')
test_hf.set_format('torch')

print(train_hf)
print(f'Features : {train_hf.features}')


---
## ⚡ CELLULE 17b — DeBERTa-v3-large : Modèle + Métriques + Trainer

In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1  = f1_metric.compute(predictions=preds, references=labels, average='weighted')['f1']
    return {'accuracy': acc, 'f1_weighted': f1}

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_NAME,
    num_labels=3,
    id2label={0:'négatif', 1:'neutre', 2:'positif'},
    label2id={'négatif':0, 'neutre':1, 'positif':2},
    ignore_mismatched_sizes=True,
    local_files_only=True
)
deberta_model.to(device)
print(f'✅ Paramètres DeBERTa : {deberta_model.num_parameters():,}')

deberta_args = TrainingArguments(
    output_dir='./deberta_results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=1e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    logging_dir='tensorboard_logs/deberta',
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=4,
    report_to='tensorboard',
    seed=42
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    tokenizer=deberta_tokenizer,
    data_collator=DataCollatorWithPadding(
        tokenizer=deberta_tokenizer, pad_to_multiple_of=8
    ),
    compute_metrics=compute_metrics
)
print('✅ Trainer prêt')


---
## ⚡ CELLULE 17c — DeBERTa-v3-large : Entraînement + Eval + Save

In [ ]:
deberta_trainer.train()

deberta_results = deberta_trainer.predict(test_hf)
preds_deberta   = np.argmax(deberta_results.predictions, axis=-1)

print(f'\nDeBERTa — Acc: {accuracy_score(y_test, preds_deberta):.4f} | '
      f'F1: {f1_score(y_test, preds_deberta, average="weighted"):.4f}')
print(classification_report(y_test, preds_deberta,
                             target_names=['négatif','neutre','positif']))

deberta_trainer.save_model('./best_deberta_model')
deberta_tokenizer.save_pretrained('./best_deberta_model')
print('✅ Sauvegardé dans ./best_deberta_model/')


---
## 🦙 CELLULE 18 — Mistral-7B QLoRA (2 pts)
> Mistral-7B a 7 milliards de paramètres. On utilise QLoRA : quantification 4-bit (le modèle tient en VRAM) + LoRA (Low-Rank Adaptation) qui n'entraîne que de petites matrices ajoutées aux couches d'attention — environ 0.1% des paramètres totaux. C'est l'état de l'art du fine-tuning efficace de LLMs.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MISTRAL_NAME = 'mistralai/Mistral-7B-v0.1'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

mistral_tokenizer              = AutoTokenizer.from_pretrained(MISTRAL_NAME, use_fast=True)
mistral_tokenizer.pad_token    = mistral_tokenizer.eos_token
mistral_tokenizer.padding_side = 'right'

def mistral_tokenize(batch):
    return mistral_tokenizer(
        batch['text'], truncation=True, max_length=256, padding=False
    )

def df_to_mistral_dataset(dataframe):
    ds = Dataset.from_dict({
        'text':  dataframe['avis_en'].astype(str).tolist(),
        'label': dataframe['label'].astype(int).tolist()
    })
    return ds.map(mistral_tokenize, batched=True, batch_size=128,
                  remove_columns=['text'], desc='Tokenizing Mistral')

train_mistral = df_to_mistral_dataset(train_df)
val_mistral   = df_to_mistral_dataset(val_df)
test_mistral  = df_to_mistral_dataset(test_df)
train_mistral.set_format('torch')
val_mistral.set_format('torch')
test_mistral.set_format('torch')

mistral_base = AutoModelForSequenceClassification.from_pretrained(
    MISTRAL_NAME,
    num_labels=3,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    ignore_mismatched_sizes=True
)
mistral_base.config.pad_token_id = mistral_tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
    target_modules=['q_proj','v_proj','k_proj','o_proj'],
    lora_dropout=0.05, bias='none'
)
mistral_model = get_peft_model(mistral_base, lora_config)
mistral_model.print_trainable_parameters()

mistral_args = TrainingArguments(
    output_dir='./mistral_results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False, bf16=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    logging_dir='tensorboard_logs/mistral',
    logging_steps=25,
    report_to='tensorboard',
    optim='paged_adamw_8bit',
    seed=42
)

mistral_trainer = Trainer(
    model=mistral_model,
    args=mistral_args,
    train_dataset=train_mistral,
    eval_dataset=val_mistral,
    tokenizer=mistral_tokenizer,
    data_collator=DataCollatorWithPadding(
        tokenizer=mistral_tokenizer, pad_to_multiple_of=8
    ),
    compute_metrics=compute_metrics  # définie en cellule 17b
)

mistral_trainer.train()

mistral_results = mistral_trainer.predict(test_mistral)
preds_mistral   = np.argmax(mistral_results.predictions, axis=-1)

print(f'\nMistral-7B QLoRA — Acc: {accuracy_score(y_test, preds_mistral):.4f} | '
      f'F1: {f1_score(y_test, preds_mistral, average="weighted"):.4f}')

mistral_model.save_pretrained('./best_mistral_lora')
mistral_tokenizer.save_pretrained('./best_mistral_lora')
print('✅ Sauvegardé dans ./best_mistral_lora/')


---
## 💾 CELLULE CHECKPOINT — À exécuter IMMÉDIATEMENT après la 18
> On sauvegarde absolument tout : splits, labels, modèles, prédictions, tokenizers, LDA. La prochaine fois qu'on relance le kernel, on met RESTORE=True tout en haut et on repart directement à la cellule 19.

In [ ]:
os.makedirs('./checkpoints', exist_ok=True)

train_df.to_csv('./checkpoints/train_df.csv', index=False)
val_df.to_csv(  './checkpoints/val_df.csv',   index=False)
test_df.to_csv( './checkpoints/test_df.csv',  index=False)

np.save('./checkpoints/y_train.npy', y_train)
np.save('./checkpoints/y_val.npy',   y_val)
np.save('./checkpoints/y_test.npy',  y_test)

with open('./checkpoints/tfidf_vectorizer.pkl', 'wb') as f: pickle.dump(tfidf, f)
with open('./checkpoints/tfidf_results.pkl',    'wb') as f:
    pickle.dump({k: {'preds': v['preds']} for k, v in tfidf_results.items()}, f)
with open('./checkpoints/keras_tokenizer.pkl',  'wb') as f: pickle.dump(keras_tok, f)
with open('./checkpoints/lda_model.pkl',        'wb') as f: pickle.dump(lda, f)
with open('./checkpoints/tfidf_lda_vectorizer.pkl','wb') as f: pickle.dump(tfidf_lda, f)
with open('./checkpoints/topic_labels.pkl',     'wb') as f: pickle.dump(topic_labels, f)

np.save('./checkpoints/preds_emb.npy',     preds_emb)
np.save('./checkpoints/preds_lstm.npy',    preds_lstm)
np.save('./checkpoints/preds_deberta.npy', preds_deberta)
np.save('./checkpoints/preds_mistral.npy', preds_mistral)

emb_model.save( './checkpoints/emb_model.keras')
lstm_model.save('./checkpoints/lstm_model.keras')

df.to_csv('./checkpoints/df_full.csv', index=False)

print('✅ CHECKPOINT COMPLET')
print(f'   preds_deberta : {preds_deberta.shape}')
print(f'   preds_mistral : {preds_mistral.shape}')


---
## 📊 CELLULE 19 — Comparaison tous les modèles (2 pts)
> On compile les performances de tous les modèles sur les mêmes métriques. C'est l'étape de comparaison qui justifie le choix du meilleur modèle et montre le gain apporté par chaque niveau de complexité — des classifieurs linéaires jusqu'au LLM à 7B paramètres.

In [ ]:
all_preds = {
    'TF-IDF + LR':      tfidf_results['Logistic Regression']['preds'],
    'TF-IDF + SVM':     tfidf_results['Linear SVM']['preds'],
    'TF-IDF + RF':      tfidf_results['Random Forest']['preds'],
    'CNN Emb. Layer':   preds_emb,
    'Bi-LSTM + GloVe':  preds_lstm,
    'DeBERTa-v3-large': preds_deberta,
    'Mistral-7B QLoRA': preds_mistral,
}

comparison = []
for name, preds in all_preds.items():
    comparison.append({
        'Modèle':        name,
        'Accuracy':      round(accuracy_score(y_test, preds), 4),
        'F1 (weighted)': round(f1_score(y_test, preds, average='weighted'), 4),
        'F1 (macro)':    round(f1_score(y_test, preds, average='macro'), 4),
    })

comp_df = pd.DataFrame(comparison).sort_values('F1 (weighted)', ascending=False)
print(comp_df.to_string(index=False))

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(comp_df)))
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

bars = axes[0].barh(comp_df['Modèle'], comp_df['Accuracy'], color=colors[::-1])
axes[0].set_xlim(0, 1)
axes[0].set_title('Accuracy par modèle', fontsize=14)
for bar, val in zip(bars, comp_df['Accuracy']):
    axes[0].text(val+0.005, bar.get_y()+bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10)

bars2 = axes[1].barh(comp_df['Modèle'], comp_df['F1 (weighted)'], color=colors[::-1])
axes[1].set_xlim(0, 1)
axes[1].set_title('F1-score pondéré par modèle', fontsize=14)
for bar, val in zip(bars2, comp_df['F1 (weighted)']):
    axes[1].text(val+0.005, bar.get_y()+bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()
comp_df.to_csv('model_comparison.csv', index=False)


---
## 🔬 CELLULE 20 — Analyse d'erreurs (1 pt)
> L'analyse d'erreurs c'est regarder ce que le modèle a raté et pourquoi. On examine la matrice de confusion et les faux positifs/négatifs. Est-ce qu'il confond 'neutre' et 'positif' ? Y a-t-il des patterns dans les erreurs ? C'est ce qui justifie des améliorations futures.

In [ ]:
best_preds  = preds_deberta
label_names = ['négatif','neutre','positif']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cm     = confusion_matrix(y_test, best_preds)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:,np.newaxis]

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=label_names, yticklabels=label_names)
axes[0].set_title('Matrice de confusion (DeBERTa)')
axes[0].set_ylabel('Vrai label'); axes[0].set_xlabel('Prédit')

sns.heatmap(cm_pct, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=label_names, yticklabels=label_names)
axes[1].set_title('Matrice normalisée (%)')
axes[1].set_ylabel('Vrai label'); axes[1].set_xlabel('Prédit')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(y_test, best_preds, target_names=label_names))

test_copy           = test_df.copy().reset_index(drop=True)
test_copy['pred']   = best_preds
test_copy['correct']= (test_copy['label'] == test_copy['pred'])
errors = test_copy[~test_copy['correct']]

print(f'Erreurs : {len(errors)}/{len(test_copy)} ({len(errors)/len(test_copy)*100:.1f}%)')
print(f'\nErreurs par label réel:')
print(errors['label'].value_counts().sort_index())

print('\n5 exemples d\'erreurs :')
for _, row in errors.sample(min(5,len(errors)), random_state=42).iterrows():
    print(f'  Vrai: {label_names[int(row["label"])]} | Prédit: {label_names[int(row["pred"])]}')
    print(f'  Avis: {str(row["avis_en"])[:100]}...')


---
## 🏢 CELLULE 21 — Analyse par assureur & thèmes (2 pts)
> On croise sentiments + assureurs + topics LDA pour produire des insights business. On répond à : quel assureur a le meilleur score sur le thème 'service client' ? Quels assureurs concentrent les avis négatifs sur le 'remboursement sinistre' ? C'est l'application concrète du NLP à l'analyse métier.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

avg_note = df.groupby('assureur')['note'].agg(['mean','count']).round(2).sort_values('mean', ascending=False)
bars = axes[0,0].bar(avg_note.index, avg_note['mean'], color='steelblue', edgecolor='black')
axes[0,0].set_ylim(0, 5.5)
axes[0,0].set_title('Note moyenne par assureur')
axes[0,0].tick_params(axis='x', rotation=45)
for bar, (_, row) in zip(bars, avg_note.iterrows()):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                   f'{row["mean"]:.2f}\n(n={int(row["count"])})', ha='center', fontsize=9)

sent_by_ass = df.groupby(['assureur','sentiment_str']).size().unstack(fill_value=0)
sent_pct    = sent_by_ass.div(sent_by_ass.sum(axis=1), axis=0)
sent_pct.plot(kind='bar', ax=axes[0,1], stacked=True,
              color=['tomato','gold','mediumseagreen'])
axes[0,1].set_title('Répartition sentiment par assureur')
axes[0,1].tick_params(axis='x', rotation=45)

topic_sent = df.groupby(['topic_label','sentiment_str']).size().unstack(fill_value=0)
topic_pct  = topic_sent.div(topic_sent.sum(axis=1), axis=0)
topic_pct.plot(kind='barh', ax=axes[1,0], stacked=True,
               color=['tomato','gold','mediumseagreen'])
axes[1,0].set_title('Sentiment par topic')

pivot = df.groupby(['assureur','topic_label'])['note'].mean().unstack(fill_value=np.nan)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=1, vmax=5, ax=axes[1,1])
axes[1,1].set_title('Note moyenne assureur × topic')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('insurer_analysis.png', dpi=150)
plt.show()


---
## 💾 CELLULE 22 — ⬛ Traduction skippée + Sauvegarde assets Streamlit
> On sauvegarde tous les fichiers nécessaires à l'app Streamlit. La traduction est intentionnellement skippée — on travaille directement sur `avis_en`.

In [ ]:
# ================================================================
# TRADUCTION — SKIPPÉE INTENTIONNELLEMENT
# On travaille directement sur la colonne `avis_en` (anglais)
# ================================================================

df.to_csv('reviews_with_topics_sentiment.csv', index=False)

with open('tfidf_vectorizer.pkl', 'wb') as f: pickle.dump(tfidf, f)
with open('logreg_model.pkl',     'wb') as f: pickle.dump(tfidf_models['Logistic Regression'], f)
with open('keras_tokenizer.pkl',  'wb') as f: pickle.dump(keras_tok, f)
with open('topic_labels.pkl',     'wb') as f: pickle.dump(topic_labels, f)
with open('lda_model.pkl',        'wb') as f: pickle.dump(lda, f)
with open('tfidf_lda_vectorizer.pkl','wb') as f: pickle.dump(tfidf_lda, f)

df[['avis_en','assureur','produit','note','sentiment_str','topic_label']].to_csv(
    'reviews_for_streamlit.csv', index=False
)
comp_df.to_csv('model_comparison.csv', index=False)

print('✅ Assets Streamlit sauvegardés')
print('Fichiers : reviews_for_streamlit.csv, tfidf_vectorizer.pkl,')
print('           logreg_model.pkl, keras_tokenizer.pkl')
print('           review_embeddings.npy, faiss_reviews.index')
print('           best_deberta_model/, best_mistral_lora/')


---
## 🚀 CELLULE 23 — Lancement Streamlit

In [ ]:
print('=== LANCER STREAMLIT ===')
print('Dans le terminal SSH :')
print('  source ~/ESILV_NPL_PROJECT/venv_nlp/bin/activate')
print('  streamlit run streamlit_app.py --server.port 8501')
print()
print('Depuis ta machine locale :')
print('  ssh -L 8501:localhost:8501 user@ton_serveur')
print('  Puis ouvre : http://localhost:8501')
print()
print('TensorBoard (optionnel) :')
print('  tensorboard --logdir=tensorboard_logs --port=6006')
print('  ssh -L 6006:localhost:6006 user@ton_serveur')
